# Topic 7 그림 코드 모음 — 강의용 Colab 데모

**확률통계 · Topic 7 · The Gaussian Distribution**

슬라이드 [T07_slides_v2.md](slides/T07_slides_v2.md) 에 쓴 그림 3개를 만드는 코드를 모았다.
슬라이드는 고정된 PNG지만, 이 노트북은 **강의 중 숫자를 바꿔가며 즉석에서 다시 그려볼 수 있다.**

| 그림 | 슬라이드 | 원본 스크립트 |
|---|---|---|
| ① 평평한 분포를 평균 냈더니 종 모양이 나온다 | 6쪽 `[S]` 평평한 분포에서 뽑아 평균만 낸다 | `figs_src_v2/t07v2_mystery.py` |
| ② 68–95–99.7 규칙과 표준화 | 12쪽 `[C]` 68–95–99.7, 그리고 겹침 | `figs_src_v2/t07v2_gaussian_rule.py` |
| ③ 히스토그램은 비슷한데 QQ plot은 다르다 | 22쪽 `[C]` 히스토그램만 보면 속는다 | `figs_src_v2/t07v2_fat_tail.py` |

⚠️ 실제 슬라이드 그림은 Windows 로컬에서 `Malgun Gothic`으로 렌더했다. 이 노트북은 Colab(Linux)이라
나눔고딕을 대신 설치해서 쓴다 — 글꼴만 다르고 배치·색·수치는 슬라이드와 동일하다.

⚠️ ① 은 이 시점에는 "정규분포"라는 이름도, CLT라는 단어도 쓰지 않는다. 관찰만 남기고
답은 Topic 10에서 확인한다 — 이 노트북도 그 순서를 그대로 따른다.

**맨 위 설정 셀을 한 번 실행한 뒤, 그림 셀은 순서와 상관없이 원하는 것만 실행하면 된다.**

In [ ]:
# 설정: 한글 글꼴 + 스타일 (맨 처음 한 번만 실행)
# Colab은 Linux라 한글 글꼴이 기본으로 없다. 나눔고딕을 설치해 등록한다.
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm, probplot, t

try:
    for f in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(f)
    KOREAN_FONT = "NanumGothic"
except Exception:
    KOREAN_FONT = "DejaVu Sans"  # 설치 실패 시 - 한글은 깨지지만 그림은 그려진다
    print("나눔고딕 설치/등록 실패 - DejaVu Sans로 대신한다 (한글이 깨질 수 있음)")

# 슬라이드 테마와 같은 색 (assets/deckstyle.py 와 동일)
C = {
    "accent": "#3b4fd8", "teal": "#0d9488", "orange": "#d97d17",
    "purple": "#8b5cf6", "pink": "#d9457f", "ink": "#0f172a",
    "body": "#4b5768", "muted": "#97a3b6", "line": "#e6eaf1", "soft": "#f7f9fc",
}
CYCLE = [C["accent"], C["teal"], C["orange"], C["purple"], C["pink"], C["muted"]]

mpl.rcParams.update({
    "font.family": KOREAN_FONT,
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.5,
    "ytick.labelsize": 12.5,
    "legend.fontsize": 12.5,
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": C["line"],
    "grid.linewidth": 1.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#d7dce6",
    "figure.dpi": 110,
})

## ① 평평한 분포를 평균 냈더니 종 모양이 나온다

$U(0,1)$ 에서 $n$ 개씩 뽑아 평균을 구한다. `TRIALS` 번 반복해서 그 평균값들의 히스토그램을 본다.
남길 관찰은 두 가지다 — ⓐ 원래 분포는 평평한데(균등) 평균을 내면 가운데가 볼록해진다,
ⓑ $n$ 이 커질수록 폭이 좁아진다. `NS` 리스트를 바꿔가며 다시 확인해볼 수 있다.

In [ ]:
TRIALS = 100_000
NS = [1, 2, 5, 30]
COLS = [C["muted"], C["orange"], C["teal"], C["accent"]]
rng = np.random.default_rng(20260302)          # 개강일 시드 (전 Topic 공통)

fig, axes = plt.subplots(1, 4, figsize=(12.0, 3.1), sharex=True)

for ax, n, col in zip(axes, NS, COLS):
    means = rng.random((TRIALS, n)).mean(axis=1)
    sd = means.std()
    print(f"n={n:>3}  평균 {means.mean():.4f}  표준편차 {sd:.4f}"
          f"   (이론 {0.2887 / np.sqrt(n):.4f})")

    ax.hist(means, bins=60, range=(0, 1), density=True, color=col,
            alpha=0.85, edgecolor="white", linewidth=0.3)
    title = "원래 분포\n(균등, n = 1)" if n == 1 else f"{n}개씩 뽑아 평균"
    ax.set_title(title, color=C["ink"], fontsize=13.5)
    ax.text(0.5, 0.93, f"표준편차 {sd:.3f}", transform=ax.transAxes,
            ha="center", va="top", fontsize=13, fontweight="bold", color=col)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([])
    ax.grid(False)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel("평균값")

fig.subplots_adjust(wspace=0.12)
plt.show()

## ② 68–95–99.7 규칙과 표준화

왼쪽: 표준정규분포에서 1σ · 2σ · 3σ 안의 넓이. 오른쪽: 평균·표준편차가 제각각인
세 분포도 표준화하면 하나로 겹친다.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.5))

# ── 왼쪽: 68-95-99.7 ───────────────────────────────────────
z = np.linspace(-4, 4, 600)
pdf = norm.pdf(z)
ax1.plot(z, pdf, color=C["ink"], linewidth=2.0)

BANDS = [(3, C["accent"], 0.12), (2, C["accent"], 0.18), (1, C["accent"], 0.30)]
for k, col, a in BANDS:
    m = np.abs(z) <= k
    ax1.fill_between(z[m], 0, pdf[m], color=col, alpha=a)

for k, y, col in [(1, 0.16, "#ffffff"), (2, 0.048, C["ink"]), (3, 0.012, C["ink"])]:
    p = norm.cdf(k) - norm.cdf(-k)
    print(f"±{k}σ 안의 넓이 {p * 100:.2f}%")
    ax1.text(0, y, f"±{k}σ : {p * 100:.1f}%", ha="center", va="center",
             fontsize=13, fontweight="bold", color=col)

ax1.set_title("68 – 95 – 99.7 규칙", color=C["ink"])
ax1.set_xlabel("z  =  표준편차 몇 개만큼")
ax1.set_ylabel("밀도")
ax1.set_xticks([-3, -2, -1, 0, 1, 2, 3])
ax1.set_ylim(0, 0.46)

# ── 오른쪽: 표준화하면 겹친다 ──────────────────────────────
PARAMS = [(174, 6, C["orange"], "키  N(174, 6²)"),
          (42, 5, C["teal"], "조립시간  N(42, 5²)"),
          (0, 1, C["accent"], "표준정규  N(0, 1²)")]
x = np.linspace(-4.2, 4.2, 600)
# 세 곡선이 정확히 포개지므로, 굵기를 달리해 "셋"임을 눈으로 보이게 한다
for (mu, sd, col, name), lw in zip(PARAMS, [7.5, 4.0, 1.8]):
    ax2.plot(x, norm.pdf(x), color=col, linewidth=lw, alpha=0.95, label=name)
ax2.text(0, 0.487, "표준화하면 완전히 포개진다", ha="center", va="top",
         fontsize=13.5, fontweight="bold", color=C["ink"])
ax2.legend(loc="upper left", fontsize=11)

ax2.set_title("$z = (x - \\mu)\\,/\\,\\sigma$ 로 옮기면", color=C["ink"])
ax2.set_xlabel("z")
ax2.set_ylabel("밀도")
ax2.set_ylim(0, 0.50)

plt.tight_layout()
plt.show()

## ③ 히스토그램은 비슷한데 QQ plot은 다르다

왼쪽: 일간 수익률 히스토그램에 정규 곡선을 겹치면 **꽤 비슷해 보인다.** 오른쪽: 같은 데이터의
QQ plot — 양 끝이 직선에서 크게 벗어난다.

⚠️ 수익률은 t(df=3) 으로 **합성**한다. 실제 주가 데이터는 저작권·재현성 문제가 있고,
t(3) 이 일간 수익률의 두꺼운 꼬리를 잘 재현한다. `DF` 를 키우면(예: 30) 꼬리가 정규분포에
가까워지는 것도 확인할 수 있다.

In [ ]:
N = 3000
DF = 3
rng = np.random.default_rng(20260302)

raw = t.rvs(DF, size=N, random_state=rng)
ret = raw / raw.std() * 0.012          # 일간 수익률 스케일(표준편차 1.2%)
z = (ret - ret.mean()) / ret.std()

for k in (3, 5):
    emp = (np.abs(z) > k).mean()
    theo = 2 * (1 - norm.cdf(k))
    print(f"|z| > {k}   실제 {emp * 100:.3f}%   정규 {theo * 100:.5f}%"
          f"   비율 {emp / theo:>8.1f}배   실제 건수 {int((np.abs(z) > k).sum())}일")
print(f"정규분포라면 |z|>5 는 약 {1 / (2 * (1 - norm.cdf(5))) / 252:,.0f}년에 한 번")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 3.2))

# ── 왼쪽: 히스토그램 + 정규 곡선 ───────────────────────────
ax1.hist(ret * 100, bins=80, range=(-6, 6), density=True, color=C["accent"],
         alpha=0.72, edgecolor="white", linewidth=0.3, label="관측 수익률")
xs = np.linspace(-6, 6, 400)
ax1.plot(xs, norm.pdf(xs, ret.mean() * 100, ret.std() * 100), color=C["ink"],
         linewidth=2.2, label="같은 평균·표준편차의 정규분포")
ax1.set_title("히스토그램 — 꼬리가 눈에 안 띈다", color=C["ink"], fontsize=14)
ax1.set_xlabel("일간 수익률 (%)")
ax1.set_ylabel("밀도")
ax1.legend(loc="upper right", fontsize=11)
for sgn in (-1, 1):                       # 문제가 숨어 있는 구간을 옅게 표시
    ax1.axvspan(sgn * 3.4, sgn * 6, color=C["orange"], alpha=0.16)
ax1.text(0, -0.30, "양 끝 옅은 구간에 문제가 숨어 있다 — 밀도가 낮아 눈에 안 보인다",
         transform=ax1.get_xaxis_transform(), ha="center", va="top",
         fontsize=11.5, color=C["orange"], fontweight="bold")

# ── 오른쪽: QQ plot ───────────────────────────────────────
(osm, osr), _ = probplot(z, dist="norm", fit=False), None
ax2.plot(osm, osr, "o", color=C["orange"], markersize=3.2, alpha=0.7)
lim = 4.2
ax2.plot([-lim, lim], [-lim, lim], color=C["ink"], linewidth=2.0, linestyle="--")
ax2.set_title("QQ plot — 양 끝이 휘어 있다", color=C["ink"], fontsize=14)
ax2.set_xlabel("정규분포라면 나왔을 값")
ax2.set_ylabel("실제 값 (표준화)")
ax2.set_xlim(-lim, lim)
# 극단값 몇 개가 세로 범위를 지배해 가운데가 뭉개진다. 범위를 잘라 준다.
YL = 9
n_out = int((np.abs(osr) > YL).sum())
ax2.set_ylim(-YL, YL)
ax2.annotate("정규분포보다\n훨씬 멀리 간다", xy=(3.2, 5.2), xytext=(0.4, 7.8),
             fontsize=12.5, fontweight="bold", color=C["orange"], ha="center",
             arrowprops=dict(arrowstyle="-|>", color=C["orange"], linewidth=1.5))
ax2.text(-lim + 0.15, -YL + 0.6, f"화면 밖 극단값 {n_out}개",
         fontsize=11.5, color=C["muted"])

plt.tight_layout()
plt.show()

---
이 노트북은 채점 대상이 아니다. 슬라이드 그림을 재생성하는 [figs_src_v2/](figs_src_v2/) 스크립트가
원본이며, 슬라이드 PNG를 바꾸려면 그쪽을 고치고 다시 실행해야 한다 — 이 노트북은 강의 중
라이브 데모·질의응답용 사본이다. QQ plot을 직접 그리는 실습은
[lab/T07_lab.ipynb](lab/T07_lab.ipynb) 에 있다.